In [ ]:
# uv pip install torch 
# . env 파일에 api key 받아놓기 

# import

In [2]:
import torch #torch 전체를 불러옴
from PIL import Image # PIL 패키지에서 Image 기능만 불러옴
from torchvision import models #

# API key 받아오기

In [4]:
from dotenv import load_dotenv #uv add python-dotenv
import os 

load_dotenv() #.env 파일을 읽음
api_key = os.getenv("ROBOFLOW_API_KEY")

C:\plantcare\
├── src\              ← 내가 만드는 코드가 전부 여기
│   ├── data\         ← 데이터 관련 코드
│   ├── models\       ← AI 모델 관련 코드
│   ├── inference\    ← 모델을 사용하는 코드 (추론)
│   ├── api\          ← 웹 서버 코드
│   └── frontend\     ← 화면(UI) 코드
├── data\             ← 데이터 파일 (이미지들)
├── models\           ← 학습된 모델 파일 (.pth)
├── docs\             ← 문서
└── .env              ← API 키

# 데이터 개수 확인

In [9]:
# 위 경로가 안 될 경우 이 경로를 시도해 보세요.
ds = load_dataset("not-lain/plantvillage")
print(len(ds["train"]))

DatasetNotFoundError: Dataset 'not-lain/plantvillage' doesn't exist on the Hub or cannot be accessed.

# 이미지 확인하기

In [ ]:
# 첫 번째 이미지를 직접 보기
item = ds["train"][0]
print(type(item["image"])) #어떤 형태인지 확인
print(item["label"])

# 이미지를 화면에 표시
item["image"] #jupyter에서는 이것만으로 이미지가 보여짐 

# EfficientNet에 이미지를 넣으면 뭔가가 나옴

In [ ]:
import torch # AI 모델을 돌리는 엔진 
from torchvision import models, transforms 

# models: 다른 사람이 만들어 놓은 AI 모델을 가져오는 곳
# transforms: 이미지를 모델이 이해할 수 있는 형태로 바꾸는 도구 
# Image : 이미지를 열고 조작하는 도구 
from PIL import Image

# 1.  사전학습된 모델 불러오기

In [ ]:

# 다른 사람이 100만 장 이미지로 미리 학습해놓은 모델을 가져옴 (전이학습)

model = models.efficientnet_b3(prtrained=True)
model.eval() # 지금은 학습이 아니라 테스트 모드야 라고 알려줌 


# 2. 이미지를 모델이 이해할 수 있는 형태로 바꾸기

In [ ]:

# 모델은 사진을 직접 못 읽음. 숫자 배열(텐서)로 바꿔야 함. 

transforms = transforms.Compose([ # transforms.Compose는 이 변환들을 순서대로 해줘. 라는 뜻. 
    transforms.Resize((224, 224)), EfficientNet이 224*224 크기 입력을 기대하기 때문 
    transforms.ToTensor(),
    transforms.Normalize( # Normalize 하는 이유는 모델이 학습할 때 이 범위의 숫자를 봤기 때문에, 같은 범위로 맞춰줘야 제대로 인식함. 
        [0.485, 0.456, 0.406],           # (이 숫자들은 ImageNet 데이터의 평균값)
        [0.229, 0.224, 0.225]
    )
])

# 3. 실제로 넣어보기
- [3, 224, 224] 의 의미: 3은 색상 채널
- 224, 224 : 가로 세로 픽셀 수

In [ ]:
img = ds["train"][0]["image"] # 아까 불러온 잎 사진
img_tensor = transform(img) # 변환 적용
print(img_tensor.shape) #torch.size([3, 224, 224])

# 4. 모델에 넣기

In [ ]:

img_tensor = img_tensor.unsqueeze(0) #unsqueeze(0)은 1장 짜리라도 1장 짜리 묶음이라고 해줘야 하니까
print(img_tensor.shape)

In [ ]:
output = model(img_tensor) # 모델에 넣기
print(output.shape) # torch.Size([1, 1000])

# 1000은 ImageNet의 클래스 수. 
# 식물 병변 7가지를 분류하고 싶으니까, 마지막 층을 바꿔야 함. 
# 

# 마지막 층 바꾸기

In [ ]:
import torch.nn as nn

# 마지막 분류층을 7클래스로 교체
model.classifier[1] = nn.Linear(1536, 7) 

output = model(img_tensor)
print(output.shape) # 1536은 print(model.classifier)을 하면 나옴. 

## EasyOCR로 한국어 텍스트를 읽을 ㅅ ㅜ있는가. 

In [ ]:
import easyocr

# OCR 모델 준비 
reader = easyocr.Reader(['ko','en']) # 한국어 영어 인식

# 사진에서 글자 읽기
results = reader.readtext('약제.jpg')

# 결과 출력
for detection in results:
    text = detection[1] #인식된 글자
    confidence = detection[2]  # 확신도 (0~1, 1에 가까울수록 정확함)
    print(f"{text} (확신도: {confidence:.2f})")

## Whisper로 한국어 음성을 인식할 수 있는가

In [ ]:
import whisper 

# 모델 로드 (base는 작은 모델, 속도 빠르고 정확도는 보통)
model = whisper.load_model("base")

# 음성 -> 텍스트 변환
result = model.transcribe("test.m4a", language="ko")
print(result["text"])



# SAM으로 잎에서 병변 영역 분리

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
import matplotlib.pyplot as plt

# SAM 모델 로드
sam = sam_model_registry["vit_b"](checkpoint="sam_vit_b_01ec64.pth")
mask_generator = SamAutomaticMaskGenerator(sam)

# 이미지 읽기
image = cv2.imread("아픈잎.jpg")
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # BGR -> RGB 변환 

# 마스크 생성
masks = mask_generator.generate(image_rgb)
print(f"마스크 {len(masks)}개 생성됨")

# 첫번째 마스크를 이미지 위에 겹쳐서 보기
plt.figure(figsize=(10, 10))
plt.imshow(image_rgb)
plt.imshow(masks[0]["segmentation"], alpha=0.5, cmap='Greens')
plt.title("SAM 세그멘테이션 결과")

# ollama로 케어 가이드 생성

In [ ]:
import requests
response = requests.post(
    "http://localhost:113434/api/generate",
    json={
        "model" : "qwen2.5:7b",
        "prompt" : "몬스테라 잎에 흰가루병이 23% 감염됨. 초기 단계. 맞춤 케어 가이드를 한국어로 작성해줘.",
        "stream" : False
    }
)

print(response.json()["response"])

# gTTS로 텍스트를 음성으로 바꿀 수 있는가

In [ ]:
from gtts import gTTS

text ="감염된 잎을 제거하고, 만코제브 수화제를 500배 희석하여 7일 간격으로 살포하세요."
tts = gTTS(text=text, lang='ko')
tts.save("케어가이드.mp3")
print("저장 완료! 케어가이드.mp3를 재생해보세요.")


: 